## Init

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim 
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType

## Reading from bronze table 

In [0]:
df=spark.table("workspace.bronze.crm_cust_info")

In [0]:
df.display()

In [0]:
df.show(5)

## Data Transformations 

In [0]:
# Checking null values 
df_nulls= df.filter(df.cst_firstname.isNull())
df_nulls.show()

In [0]:
# trimming the values 
# for any string in the data frame, trim the string

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df=df.withColumn(field.name, trim(col(field.name)))

In [0]:
# Checking the result
df.display()

In [0]:
# Normalization : gender

df=df.withColumn("cst_gndr",F.when(df.cst_gndr=="M","Male").when(df.cst_gndr=="F","Female").otherwise("n/a"))
df.display()

In [0]:
# Normalization : marital status 
df=df.withColumn("cst_marital_status",F.when(df.cst_marital_status=="M","Married").when(df.cst_marital_status=="S","Single").otherwise("n/a"))
df.display()


In [0]:
# Changing the column names 

RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

In [0]:
df_filter=df.filter(df.customer_id.isNull())
df_filter.show(5)

In [0]:
# Dropping  Null customer_id
df=df.filter(df.customer_id.isNotNull())
df.show(5)

In [0]:
# Count nulls for each column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

In [0]:
# Deleting nulls 
df=df.filter(df.first_name.isNotNull())
df=df.filter(df.last_name.isNotNull())

df.show(5)

In [0]:
# Checking for nulls 
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

In [0]:
# Capitalize first and last names properly
df = df.withColumn("first_name", F.initcap("first_name")).withColumn("last_name", F.initcap("last_name"))

df.show(5)

In [0]:
# Convert created_date to PySpark DateType
df = df.withColumn("created_date", df.created_date.cast(DateType()))
df.show(5)

## Write into silver Tqble 

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("Silver.crm_cust_info")

In [0]:
%sql
-- Checking silver.crm_cust_info table 
select * from silver.crm_cust_info limit 5